# Notebook 1 · Introduction to Linear Models

Fit your first straight line to data and learn how to judge it, using one simple 11-step recipe.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/notebooks/nb-01-linear-intro.ipynb)

- Lecture: [en-01-linear-intro.md](https://github.com/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/en-01-linear-intro.md)
- Lab: [LM101](https://rathachai.github.io/DA-LAB/learnings/linear/lm101.html)

The same 11-step recipe every time:

1. Load libraries
2. Get data
3. Display data
4. Visualize correlation
5. Process data
6. Select X and y
7. Split into train and test
8. Create and train the model
9. Predict
10. Evaluate
11. Visualize y and y_pred

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io

## Step 1 · Load libraries
Like opening your toolbox: pandas for tables, numpy for arithmetic, scikit-learn for the model and the scores, matplotlib only for drawing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             mean_absolute_percentage_error, r2_score)
pd.options.display.float_format = '{:.3f}'.format

In [ ]:
print("Libraries ready")

## Step 2 · Get data
We read a CSV file straight from the web into a table called `df`. Each row is one measurement: `id` (a row label), `x` (the input we know) and `y` (the output we want to predict).

In [ ]:
url = "https://rathachai.github.io/DA-LAB/learnings/linear/data/lm101_points.csv"

In [ ]:
df = pd.read_csv(url)

In [ ]:
df

## Step 3 · Display data
Always look before you model: how big is the table, and what are typical values? Think of it as reading the nameplate before wiring a motor.

In [ ]:
print("rows, columns:", df.shape)

In [ ]:
df.describe()

## Step 4 · Visualize correlation
A scatter plot shows whether `y` rises with `x` in a straight-line way. Pearson **r** measures this from -1 to +1: near +1 means a clean rising line, near 0 means no straight-line link.

In [ ]:
r = df["x"].corr(df["y"])

In [ ]:
plt.scatter(df["x"], df["y"])
plt.title(f"y vs x   (Pearson r = {r:.3f})")
plt.xlabel("x"); plt.ylabel("y"); plt.grid(alpha=0.3); plt.show()

## Step 5 · Process data
Nothing to do: the data is clean. (`id` only names the row, so it is not a feature and we simply ignore it.)

## Step 6 · Select X and y
`X` = the input column(s) (capital, always a table), `y` = the answer column (lowercase, a single column).

In [ ]:
X = df[["x"]]

In [ ]:
y = df["y"]

In [ ]:
print("X shape:", X.shape, "| y shape:", y.shape)

In [ ]:
X

## Step 7 · Split into train and test
No split in this chapter. We use all 40 points to learn and to check; splitting comes in a later chapter.

## Step 8 · Create and train the model
The model is a straight line `y = m*x + c`. `m` (slope) is how much y changes per 1 unit of x, `c` (intercept) is y when x = 0. `fit` finds the `m` and `c` with the smallest total squared miss.

In [ ]:
model = LinearRegression()

In [ ]:
model.fit(X, y)

In [ ]:
m, c = model.coef_[0], model.intercept_

In [ ]:
print(f"slope m = {m:.3f}, intercept c = {c:.3f}")

In [ ]:
print(f"y = {m:.3f} * x + {c:.3f}")

## Step 9 · Predict
Feed every x into the line to get the predicted y (`y_pred`). The gap `y - y_pred` is the **residual**, the model's miss for that row.

In [ ]:
y_pred = model.predict(X)

In [ ]:
out = df[["x", "y"]].copy()

In [ ]:
out["y_pred"] = y_pred

In [ ]:
out["residual"] = y - y_pred

In [ ]:
out

## Step 10 · Evaluate
Four report-card numbers: **MAE** = average miss, **RMSE** = like MAE but punishes big misses (same unit as y), **MAPE** = average miss in % of y, **R²** = share of the ups and downs of y that the line explains (1 = perfect).

In [ ]:
mae = mean_absolute_error(y, y_pred)
print("MAE  =", round(mae, 3))

In [ ]:
rmse = np.sqrt(mean_squared_error(y, y_pred))
print("RMSE =", round(rmse, 3))

In [ ]:
mape = 100 * mean_absolute_percentage_error(y, y_pred)
print("MAPE =", round(mape, 2), "%")

In [ ]:
r2 = r2_score(y, y_pred)
print("R2   =", round(r2, 3))

## Step 11 · Visualize y and y_pred
Left: the fitted line with a gray stick for each residual. Right: actual vs predicted; points on the dashed diagonal `y = y_pred` would be perfect predictions.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
xs = np.sort(df["x"])
ax[0].scatter(df["x"], y, label="actual y")
ax[0].plot(xs, m * xs + c, color="tab:red", label="fitted line")
ax[0].vlines(df["x"], y_pred, y, color="gray", alpha=0.6)
ax[0].set_xlabel("x"); ax[0].set_ylabel("y"); ax[0].set_title("Fitted line and residuals"); ax[0].legend()
ax[1].scatter(y, y_pred)
lim = [y.min(), y.max()]; ax[1].plot(lim, lim, "k--", label="y = y_pred")
ax[1].set_xlabel("actual y"); ax[1].set_ylabel("predicted y"); ax[1].set_title("Actual vs predicted"); ax[1].legend()
plt.tight_layout(); plt.show()

## Bonus · How does the model find m and c?
*Optional.* `fit` uses an exact formula. A computer can also find the line by **gradient descent**: start with a guess, then repeatedly nudge `m` and `c` downhill on the error. The nudge size is the **learning rate**: too small is slow, too large overshoots and blows up (like over-correcting a steering wheel).

In [ ]:
x_val = df["x"].values
y_val = y.values

In [ ]:
lr = 0.0005   # too small
m_gd, c_gd, loss_small = 0.0, 0.0, []
for step in range(100):
    e = y_val - (m_gd * x_val + c_gd)
    loss_small.append(np.mean(e ** 2))
    m_gd += lr * 2 * np.mean(x_val * e)
    c_gd += lr * 2 * np.mean(e)
print("lr =", lr, "| m =", round(m_gd, 3), "| c =", round(c_gd, 3), "| final MSE =", round(loss_small[-1], 3))

In [ ]:
lr = 0.01   # good
m_gd, c_gd, loss_good = 0.0, 0.0, []
for step in range(100):
    e = y_val - (m_gd * x_val + c_gd)
    loss_good.append(np.mean(e ** 2))
    m_gd += lr * 2 * np.mean(x_val * e)
    c_gd += lr * 2 * np.mean(e)
print("lr =", lr, "| m =", round(m_gd, 3), "| c =", round(c_gd, 3), "| final MSE =", round(loss_good[-1], 3))

In [ ]:
lr = 0.1   # too large
m_gd, c_gd, loss_big = 0.0, 0.0, []
for step in range(100):
    e = y_val - (m_gd * x_val + c_gd)
    loss_big.append(np.mean(e ** 2))
    m_gd += lr * 2 * np.mean(x_val * e)
    c_gd += lr * 2 * np.mean(e)
print("lr =", lr, "| m =", round(m_gd, 3), "| c =", round(c_gd, 3), "| final MSE =", round(loss_big[-1], 3))

In [ ]:
print("sklearn fit  | m =", round(m, 3), "| c =", round(c, 3))

In [ ]:
plt.plot(loss_small, label="too small (0.0005)")
plt.plot(loss_good, label="good (0.01)")
plt.plot(loss_big, label="too large (0.1)")
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("MSE (log scale)")
plt.title("Loss curve for three learning rates"); plt.legend(); plt.show()

## Summary

- A table has `id` (row label), `X` (inputs) and `y` (the answer to predict).
- A linear model is `y = m*x + c`; the residual is the miss `y - y_pred`.
- MAE, RMSE, MAPE and R² tell you how good the line is.
- `LinearRegression().fit(X, y)` finds the best `m` and `c`; gradient descent reaches the same answer step by step if the learning rate is sensible.

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io